<a href="https://colab.research.google.com/github/lucas-j-zheng/ml-practice-from-scratch/blob/main/lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import RobertaModel, RobertaTokenizerFast
from sklearn.metrics import accuracy_score, matthews_corrcoef
from scipy.stats import pearsonr
import random
from datasets import load_dataset


import math


In [19]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
def set_seed(s=42):
  random.seed(s)
  np.random_seed(s)
  torch.manual_seed(s)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(s)

TASK_CONFIG = {
    'mrpc':  {'cols': ('sentence1', 'sentence2'), 'num_outputs': 2, 'is_regression': False},
    'cola':  {'cols': ('sentence',  None),         'num_outputs': 2, 'is_regression': False},
    'stsb':  {'cols': ('sentence1', 'sentence2'), 'num_outputs': 1, 'is_regression': True},
}

In [20]:
MODEL_NAME = 'roberta-base'
HIDDEN = 768

## Data Loading



In [21]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME)


In [24]:
class GLUEDataset(torch.utils.data.Dataset):
  def __init__(self, texts_a, texts_b, labels, tokenizer, is_regression):
    self.texts_a = texts_a
    self.texts_b = texts_b
    self.labels = labels
    self.tokenizer = tokenizer
    self.is_regression = is_regression

  def __len__(self):
    return len(self.labels)


  def __getitem__(self, i):
    # tokenize and return one example
    if self.texts_b is None:
      encoded = self.tokenizer(
        self.texts_a[i],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
      )
    else:
      encoded = self.tokenizer(
        self.texts_a[i], self.texts_b[i],
        add_special_tokens=True,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
      )
    label_dtype = torch.float if self.is_regression else torch.long
    label = torch.tensor(self.labels[i], dtype=label_dtype)

    return {
        'input_ids': encoded['input_ids'].squeeze(0),
        'attention_mask': encoded['attention_mask'].squeeze(0),
        'labels': label,
    }





In [25]:
def build_dataloaders(task, batch_size=32):
  cfg = TASK_CONFIG[task]
  col_a, col_b = cfg['cols']
  is_regression = cfg['is_regression']
  raw = load_dataset('glue', task)

  def make_split(split):
    a = raw[split][col_a]
    b = raw[split][col_b] if col_b is not None else None
    y = raw[split]['label']
    return GLUEDataset(a, b, y, tokenizer, is_regression)

  train = make_split('train')
  val = make_split('validation')
  test = make_split('test')

  train_dl = DataLoader(train, batch_size=batch_size, shuffle=True)
  val_dl = DataLoader(val, batch_size=batch_size, shuffle=False)

  return train_dl, val_dl

# Model Finetuning (FULL)

In [ ]:
class RobertaClassifier(nn.Module):
  def __init__(self, base, num_outputs):
    super().__init__()
    self.base = base
    self.dropout = nn.Dropout(0.1)
    self.head = nn.Linear(HIDDEN, num_outputs)

  def forward(self, input_ids, attention_mask):
    # forward pass through the base model
    x = self.base(input_ids, attention_mask)
